# 03 · Backtest walk-forward semanal

Para cada semana de 2024, 2025 y 2026 (ya jugadas): **entreno solo con los partidos anteriores, predigo esa semana y guardo la predicción junto al resultado real.** Es lo más parecido a usar el modelo en tiempo real.

- **Modelos:** LightGBM, uno para QB/RB/WR/TE (con la posición como variable), uno para K y uno para D/ST. Código en `src/fantasy_ml/model.py`.
- **Hiperparámetros:** búsqueda pequeña evaluada **solo con 2024** y congelada en `config/model.yaml`. Por eso 2024 es optimista; las temporadas fuera de muestra son **2025 y 2026**. 2026 tiene muy pocas semanas para ajustar nada con ella.
- **Contra qué se compara:** con el **baseline** (media de los últimos 5 partidos) en todas las temporadas, y con las **proyecciones de ESPN** solo en 2026, la única temporada de la liga.
- **Supuesto:** solo se evalúan partidos jugados, porque el modelo predice los puntos *si el jugador juega*.

## 1. Setup

In [ ]:
import polars as pl
import yaml

from fantasy_ml import data, espn, model as M, evaluation as E, features as F
from fantasy_ml.data import DATA_PROC, CONFIG

VALIDATION = data.load_config("validation")
TUNING_SEASON = VALIDATION["tuning_season"]
EVAL_SEASONS = [e["season"] for e in VALIDATION["evaluation"]]
OUT_OF_SAMPLE = [e["season"] for e in VALIDATION["evaluation"] if e["role"] == "out_of_sample"]

FEATURES = {"offense": pl.read_parquet(DATA_PROC / "features_offense.parquet"),
            "k": pl.read_parquet(DATA_PROC / "features_k.parquet"),
            "dst": pl.read_parquet(DATA_PROC / "features_dst.parquet")}
for g, df in FEATURES.items():
    print(f"{g:8s} {df.height:>6,} filas · {len(F.feature_cols(df))} features")
print(f"Ajuste: {TUNING_SEASON} · evaluación: {EVAL_SEASONS} · fuera de muestra: {OUT_OF_SAMPLE}")

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(20)

## 2. Hiperparámetros: búsqueda pequeña con 2024 y congelados

Pruebo 12 combinaciones: `num_leaves` ∈ {7, 15, 31} × `min_child_samples` ∈ {20, 80} × `n_estimators` ∈ {200, 500}, con `learning_rate` = 0.03. El resto es fijo (submuestreo del 80% de filas y columnas, `reg_lambda` = 1).

Cada combinación se evalúa con el **mismo walk-forward semanal, pero solo en las 18 semanas de 2024**: se entrena con 2023 y las semanas previas de 2024. El criterio es el MAE en los jugadores relevantes para fantasy (definidos en la sección 3). La combinación ganadora se guarda en `config/model.yaml` y **no se vuelve a tocar**: 2025 y 2026 se predicen con ella.

`RETUNE = False` carga la elección guardada. Ponerlo en `True` repite la búsqueda, que tarda unos 10 minutos.

In [ ]:
RETUNE = False
GRID = {"num_leaves": [7, 15, 31], "min_child_samples": [20, 80], "n_estimators": [200, 500], "learning_rate": [0.03]}
model_cfg_path = CONFIG / "model.yaml"

if RETUNE or not model_cfg_path.exists():
    results, chosen = {}, {}
    for g, df in FEATURES.items():
        print(f"Buscando {g}...")
        res = M.search(df, g, GRID, season=TUNING_SEASON, verbose=False)
        results[g] = res
        chosen[g] = {k: (v.item() if hasattr(v, "item") else v) for k, v in res.row(0, named=True).items() if k in GRID}
    model_cfg = {
        "tuned_on": TUNING_SEASON,
        "criterion": "MAE en jugadores relevantes, walk-forward semanal de la temporada de ajuste",
        "fixed_params": M.FIXED_PARAMS,
        "grid": GRID,
        "params": chosen,
        "search_results": {g: [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}
                               for r in res.to_dicts()] for g, res in results.items()},
    }
    with model_cfg_path.open("w", encoding="utf-8") as f:
        f.write("# Generado por notebooks/03_backtest.ipynb (RETUNE=True). Hiperparámetros congelados:\n"
                "# elegidos solo con el walk-forward de la temporada `tuned_on` y usados sin cambios después.\n")
        yaml.safe_dump(model_cfg, f, allow_unicode=True, sort_keys=False)
    print(f"✓ Guardado {model_cfg_path.relative_to(data.ROOT)}")
else:
    model_cfg = yaml.safe_load(model_cfg_path.read_text(encoding="utf-8"))
    print(f"Hiperparámetros congelados (ajustados con {model_cfg['tuned_on']}), cargados de config/model.yaml")

PARAMS = model_cfg["params"]
for g in FEATURES:
    print(f"  {g:8s} {PARAMS[g]}")

In [ ]:
# Resultados de la búsqueda (MAE en 2024; menor es mejor)
pl.concat([pl.DataFrame(rows).with_columns(grupo=pl.lit(g)).head(4) for g, rows in model_cfg["search_results"].items()],
          how="diagonal_relaxed").select("grupo", *GRID, "mae_relevant", "mae_baseline_relevant", "mae_all")

## 3. Backtest semana por semana

Para cada semana evaluada: `train` = todas las filas anteriores con resultado conocido, `test` = esa semana. En total son 38 semanas por modelo.

**Jugadores relevantes:** en cada semana y posición tomo los primeros según el baseline: 24 QB, 48 RB, 60 WR, 24 TE, 20 K y 20 D/ST. Aproxima a quienes se consideran en una liga de 10 equipos y deja fuera a los suplentes con ~0 puntos, que harían que cualquier modelo pareciera bueno. El grupo lo define el baseline, no el modelo, así que es el mismo para todas las comparaciones.

In [ ]:
weeks = {g: M.eval_weeks(df, EVAL_SEASONS) for g, df in FEATURES.items()}
bt = pl.concat([M.walk_forward(df, g, PARAMS[g], weeks[g]).with_columns(group=pl.lit(g))
                for g, df in FEATURES.items()], how="diagonal_relaxed")
bt = bt.with_columns(error=pl.col("pred") - pl.col("y"), abs_error=(pl.col("pred") - pl.col("y")).abs(),
                     sample=pl.when(pl.col("season") == TUNING_SEASON).then(pl.lit("ajuste")).otherwise(pl.lit("fuera de muestra")))
bt.write_parquet(DATA_PROC / "backtest_predictions.parquet")

print(f"✓ data/processed/backtest_predictions.parquet · {bt.height:,} predicciones")
bt.group_by("season").agg(pl.col("week").n_unique().alias("semanas"), pl.len().alias("predicciones"),
                          pl.col("relevant").sum().alias("relevantes")).sort("season")

## 4. Error por posición

MAE = error absoluto medio (en puntos). **Sesgo** = predicción − real, en promedio: positivo significa que sobreestima. Comparo modelo y baseline sobre las mismas filas. 2024 está marcado porque se usó para elegir los hiperparámetros.

In [ ]:
by_pos = (E.metrics(bt.filter("relevant"), ["season", "position"])
          .with_columns(mejora_pct=((pl.col("mae_baseline") - pl.col("mae_pred")) / pl.col("mae_baseline") * 100).round(1))
          .select("season", "position", "n", "mae_pred", "mae_baseline", "mejora_pct", "rmse_pred", "rmse_baseline", "bias_pred", "bias_baseline"))
by_pos

Todas las filas (incluidos suplentes con ~0 puntos) como referencia:

In [ ]:
E.metrics(bt.filter(pl.col("season").is_in(OUT_OF_SAMPLE)), ["position"]).select("position", "n", "mae_pred", "mae_baseline", "bias_pred")

### ¿La mejora sobre el baseline es real?

Diferencia de MAE (modelo − baseline) con un intervalo de confianza del 95% por bootstrap, **remuestreando semanas completas**, porque los errores de una misma semana están correlacionados. Negativo = el modelo es mejor. Si el intervalo no cruza 0, la diferencia es consistente. **Con menos de 6 semanas (2026, por ahora) no hay suficientes semanas para un intervalo creíble**, así que solo se muestra el valor puntual.

In [ ]:
rows = []
for season in EVAL_SEASONS:
    for pos in ["QB", "RB", "WR", "TE", "K", "D/ST"]:
        sub = bt.filter("relevant", pl.col("season") == season, pl.col("position") == pos)
        if sub.height:
            rows.append({"season": season, "position": pos, **E.bootstrap_mae_diff(sub)})
    sub = bt.filter("relevant", pl.col("season") == season)
    rows.append({"season": season, "position": "TODAS", **E.bootstrap_mae_diff(sub)})
boot = pl.DataFrame(rows).with_columns(
    veredicto=E.verdict(pl.col("ci_low"), pl.col("ci_high"), pl.col("weeks"), worse="baseline mejor"))
boot

## 5. Contra ESPN (2026)

ESPN no guarda las proyecciones de semanas pasadas para todos los jugadores. Solo se pueden recuperar las de los jugadores que estaban en un roster de la liga (unos 150 por semana), desde los box scores. La comparación es sobre ese grupo, con los mismos jugadores para los tres métodos. **Son muy pocas semanas: sirve como primera señal, no como conclusión.** El registro semanal de predicciones (notebook 04) irá acumulando la comparación completa durante la temporada.

In [ ]:
weeks_2026 = sorted(bt.filter(pl.col("season") == 2026)["week"].unique().to_list())
espn_path = data.DATA_RAW / "espn_rostered_projections_2026.parquet"
cached = pl.read_parquet(espn_path) if espn_path.exists() else None
if cached is None or sorted(cached["week"].unique().to_list()) != weeks_2026:
    league = espn.connect(2026)
    cached = pl.concat([espn.rostered_projections(league, w).with_columns(week=pl.lit(w, dtype=pl.Int32)) for w in weeks_2026])
    cached.write_parquet(espn_path)
espn_proj = cached.select("week", "espn_id", "espn_projection").unique(["week", "espn_id"])

ids = F.gsis_to_espn(data.load_sources()["playerids"])
bt26 = (bt.filter(pl.col("season") == 2026)
          .join(ids.rename({"espn_id": "_espn_off"}), on="player_id", how="left")
          .with_columns(espn_id=pl.coalesce("espn_id", "_espn_off")).drop("_espn_off")
          .join(espn_proj, on=["week", "espn_id"], how="inner")
          .filter(pl.col("espn_projection").is_not_null()))

print(f"Jugadores-semana de 2026 con proyección de ESPN: {bt26.height} (semanas {weeks_2026})")
vs_espn = E.metrics(bt26, ["position"], models=("pred", "baseline", "espn_projection"))
pl.concat([vs_espn, E.metrics(bt26.with_columns(position=pl.lit("TODAS")), ["position"], models=("pred", "baseline", "espn_projection"))]) \
  .select("position", "n", "mae_pred", "mae_baseline", "mae_espn_projection", "bias_pred", "bias_espn_projection")

In [ ]:
d = E.bootstrap_mae_diff(bt26, a="pred", b="espn_projection")
print(f"MAE modelo − ESPN: {d['mae_diff']:+.2f} puntos en {d['n']} jugadores-semana ({d['weeks']} semanas)")
if d["weeks"] < E.MIN_WEEKS:
    print(f"  Solo {d['weeks']} semanas: el intervalo bootstrap no es fiable; se reevaluará con el registro semanal (notebook 04).")
else:
    print(f"  IC 95%: [{d['ci_low']:+.2f}, {d['ci_high']:+.2f}]")

## 6. ¿Dónde falla más?

Análisis sobre las temporadas **fuera de muestra (2025 y 2026)**, solo QB/RB/WR/TE relevantes, salvo donde se indica.

In [ ]:
oos = bt.filter(pl.col("season").is_in(OUT_OF_SAMPLE), "relevant")
off = oos.filter(pl.col("group") == "offense")

def seg(df, col, label):
    return (df.group_by(col).agg(pl.len().alias("n"), pl.col("abs_error").mean().round(2).alias("mae_modelo"),
                                 (pl.col("baseline") - pl.col("y")).abs().mean().round(2).alias("mae_baseline"),
                                 pl.col("error").mean().round(2).alias("sesgo"))
              .sort(col).rename({col: label}))

### Experiencia: debutantes y jugadores con pocos partidos

In [ ]:
seg(off.with_columns(exp=pl.when(pl.col("games_career") == 0).then(pl.lit("0 (debut)"))
                          .when(pl.col("games_career") <= 3).then(pl.lit("1-3"))
                          .when(pl.col("games_career") <= 16).then(pl.lit("4-16"))
                          .otherwise(pl.lit("17+"))), "exp", "partidos previos")

### Regreso tras una ausencia y momento de la temporada

In [ ]:
pl.concat([
    seg(off.with_columns(v=pl.when(pl.col("weeks_since_last").is_null()).then(pl.lit("1er partido de la temporada"))
                         .when(pl.col("weeks_since_last") == 1).then(pl.lit("jugó la semana anterior"))
                         .when(pl.col("weeks_since_last") == 2).then(pl.lit("volvió tras 1 semana (bye o ausencia)"))
                         .otherwise(pl.lit("volvió tras 2+ semanas"))), "v", "segmento"),
    seg(off.with_columns(v=pl.when(pl.col("week") <= 4).then(pl.lit("semanas 1-4")).otherwise(pl.lit("semanas 5-18"))), "v", "segmento"),
])

### Calibración

Agrupo las predicciones en 10 tramos y comparo la media predicha con la media real de cada tramo. Un modelo bien calibrado queda sobre la diagonal (predicho ≈ real).

In [ ]:
(off.with_columns(tramo=pl.col("pred").qcut(10, labels=[f"D{i}" for i in range(1, 11)]))
    .group_by("tramo").agg(pl.len().alias("n"), pl.col("pred").mean().round(2).alias("pred_media"),
                            pl.col("y").mean().round(2).alias("real_media"), pl.col("abs_error").mean().round(2).alias("mae"))
    .sort("pred_media"))

### Partidos explosivos

¿Cuánto del error se concentra en los partidos en que el jugador explota (muchos puntos, casi siempre por TDs)?

In [ ]:
(off.with_columns(resultado=pl.when(pl.col("y") < 5).then(pl.lit("a) < 5 pts"))
                            .when(pl.col("y") < 15).then(pl.lit("b) 5-15"))
                            .when(pl.col("y") < 25).then(pl.lit("c) 15-25"))
                            .otherwise(pl.lit("d) 25+")))
    .group_by("resultado").agg(pl.len().alias("n"), pl.col("abs_error").mean().round(2).alias("mae"),
                                pl.col("error").mean().round(2).alias("sesgo"),
                                (pl.col("abs_error").sum() / off["abs_error"].sum() * 100).round(1).alias("pct_del_error_total"))
    .sort("resultado"))

### Los 15 errores más grandes

In [ ]:
(oos.sort("abs_error", descending=True).head(15)
    .select("season", "week", "position", pl.coalesce("player_display_name", "team").alias("jugador"),
            pl.col("pred").round(1), pl.col("baseline").round(1), "y", pl.col("error").round(1)))

### K y D/ST por semana del año

In [ ]:
seg(oos.filter(pl.col("group") != "offense").with_columns(v=pl.concat_str("position", pl.lit(" · "),
     pl.when(pl.col("week") <= 4).then(pl.lit("semanas 1-4")).otherwise(pl.lit("semanas 5-18")))), "v", "segmento")

## 7. Conclusiones

**Contra el baseline, fuera de muestra (2025):** el modelo es mejor en todas las posiciones, con la diferencia consistente semana a semana: el MAE en jugadores relevantes baja entre un 6% (WR) y un 10% (QB, D/ST), unos 0.5 puntos por jugador y partido. La mejora de 2025 es igual a la de 2024 (la temporada de ajuste), así que la búsqueda de hiperparámetros no infló los resultados.

**2026 (2 semanas):** misma dirección, salvo en QB, donde no mejora al baseline. Son muy pocas semanas para concluir nada.

**Contra ESPN (2026, 292 jugadores-semana con roster en la liga):** **ESPN es algo mejor** (MAE 6.25 contra 6.46 del modelo). El modelo le gana en WR y empata en K y D/ST, pero pierde en RB, TE y QB. Además, el modelo **subestima** a estos jugadores (−1.4 puntos en promedio), mientras que ESPN casi no tiene sesgo. Tiene sentido: son jugadores elegidos en el draft (titulares y buenos), y al inicio de la temporada el modelo depende del año anterior, mientras que ESPN incorpora información que el modelo no tiene: cambios de equipo y de rol en la pretemporada, y noticias de lesiones y del depth chart. Con 2 semanas no es concluyente; el registro semanal dirá si se mantiene.

**Dónde falla más:**
- **Partidos explosivos:** los de 25+ puntos son el 8% de los partidos pero generan el 20% del error, y siempre se subestiman (−15 puntos), casi siempre por TDs. Es en buena parte ruido imposible de predecir. Los de menos de 5 puntos (lesión durante el partido, rol menor del esperado) son otro 29% del error.
- **Regresos tras 2+ semanas de ausencia:** el modelo sobreestima (+2.4 puntos). Asume que el jugador vuelve con su rol previo, pero suele volver con menos carga.
- **Novatos con 1–3 partidos:** sobreestimación de +1.9 puntos. En cambio, en el debut el modelo es mucho mejor que el baseline (3.5 contra 6.4 de MAE), porque usa el contexto del partido en lugar de la media de la posición.
- **Calibración:** buena en general (media predicha ≈ real por tramos), con una ligera sobreestimación, de unos 0.5–1 punto, en los tramos medios.

**Posibles mejoras** (evaluándolas siempre con el walk-forward y sin ajustar con 2026): features de rol en la pretemporada y de cambio de equipo; ajustar el regreso tras una ausencia (por ejemplo, con el % de snaps del partido de regreso en temporadas anteriores); usar la proyección de ESPN como feature una vez que el registro tenga varias semanas; y probar un modelo aún más simple, porque la búsqueda eligió el extremo de la rejilla.